# FGSM Vision Engine - Colab Training
Just run the cells below! The notebook will automatically download the dataset from your AWS S3 bucket, mount your Google Drive, and save checkpoints directly to your Drive so you don't lose them if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fgsm_runs

In [ ]:
%cd /content
# Paste a freshly-generated presigned URL here at runtime - never hardcode
# one in this notebook. Generate one with:
#   aws s3 presign s3://fgsm-vision-models-aibridix-official/fgsm_colab_package.zip --expires-in 3600 --profile aibridix_official
# A prior version of this cell had a long-lived presigned URL (and the AWS
# access key it embedded) committed directly to git, which is very likely
# what got the old AWS account suspended for credential exposure.
from getpass import getpass
package_url = getpass("Presigned S3 URL for fgsm_colab_package.zip: ")
!wget -O fgsm_colab_package.zip "{package_url}"
!unzip -o -q fgsm_colab_package.zip


In [ ]:
%cd /content/fgsm_colab_package
!pip install -r requirements.txt

In [ ]:
# Second safety net alongside Drive: periodically sync checkpoints to S3
# too, in case Drive's own sync to your account lags or glitches. Credentials
# are entered here at runtime via getpass - never hardcode them in this
# notebook (see the note on the download cell above for why that matters).
import getpass, os
os.environ["AWS_ACCESS_KEY_ID"] = getpass.getpass("AWS Access Key ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass("AWS Secret Access Key: ")
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
!pip install -q awscli
get_ipython().system_raw(
    "while true; do sleep 300; "
    "aws s3 sync /content/drive/MyDrive/fgsm_runs/ "
    "s3://fgsm-vision-models-aibridix-official/colab_runs/ --quiet; done &"
)
print("Background S3 sync started (every 5 min).")


In [ ]:
# Patch the script to fix the warmup_ratio bug, and point output_dir
# directly at Drive (instead of writing locally then cp -r'ing at the end)
# so a mid-training Colab disconnect doesn't lose everything.
!sed -i '/warmup_ratio/d' src/train_temporal_classifier.py
!sed -i 's#output_dir:.*#output_dir: /content/drive/MyDrive/fgsm_runs/temporal_move_classifier#' configs/temporal_classifier.yaml
!python src/train_temporal_classifier.py --config configs/temporal_classifier.yaml


In [ ]:
# train_hitbox_segmentation.py now checkpoints every 200 steps (not per
# epoch), so it saves durably well before completing any single epoch on
# this dataset - no need to shorten the run via max_steps anymore. It also
# resumes automatically from the latest checkpoint under output_dir, so if
# this Colab session disconnects, just re-run this cell (after re-running
# the setup cells above) and it'll pick up where it left off.
!python src/train_hitbox_segmentation.py --data data/ufd/segmentation_dataset --output /content/drive/MyDrive/fgsm_runs/hitbox_segmenter

In [ ]:
!echo 'Training complete! Checkpoints are saved in your Google Drive under MyDrive/fgsm_runs/ !'